# ShadowWork-SHIFT | Synchronize final dataset card and remove empty HF placeholder

**CPU-only, single-use release cleanup.** In Kaggle, set the write-enabled `HF_TOKEN` secret and enable Internet. This notebook first publishes only the canonical GitHub dataset card to the existing Hugging Face dataset, verifies the anonymous download and unchanged data inventory; it then removes **only** the unused model repository `i-am-shaurya05/shadowwork-shift-models`, after asserting it contains exactly `.gitattributes` and is not in the public model collection. It does not modify any trained model, model collection, evaluation trace or dataset row. After successful execution, the notebook itself can be retired from GitHub.


In [ ]:
%pip install -q "huggingface_hub>=0.34,<2" requests


In [ ]:
# Synchronize ONLY the Hugging Face dataset README.md with the current
# peer-reviewed source card. No model, training data, logs or experiment changes.
from pathlib import Path
from hashlib import sha256
from io import BytesIO
import requests
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

DATASET_ID = "i-am-shaurya05/shadowwork-shift-datasets"
CARD_SOURCE = (
    "https://raw.githubusercontent.com/githubshaurya/"
    "ShadowWork-SHIFT/main/datasets/HF_DATASET_CARD.md"
)

response = requests.get(CARD_SOURCE, timeout=40)
response.raise_for_status()
card_bytes = response.content
card_text = card_bytes.decode("utf-8")
assert "SW17 now fully released" in card_text
assert "31/80" in card_text and "65/80" in card_text
assert "Detailed creation process" in card_text and "training" in card_text

token = UserSecretsClient().get_secret("HF_TOKEN")
assert token, "Set a write-enabled HF_TOKEN Kaggle Secret"
api = HfApi(token=token)
assert api.whoami().get("name") == "i-am-shaurya05"
before = api.dataset_info(DATASET_ID)
assert not before.private
before_names = {x.rfilename for x in before.siblings}
assert "sw17/learner_corrective_v4_training.jsonl" in before_names
assert "sw17/policy_summary.csv" in before_names

api.upload_file(
    path_or_fileobj=BytesIO(card_bytes),
    path_in_repo="README.md",
    repo_id=DATASET_ID,
    repo_type="dataset",
    commit_message="Refresh dataset card with verified SW17 training attribution",
)

# Anonymous download verifies the exact published card, not a local copy.
published = hf_hub_download(
    repo_id=DATASET_ID,
    filename="README.md",
    repo_type="dataset",
    token=False,
    force_download=True,
)
assert sha256(Path(published).read_bytes()).hexdigest() == sha256(card_bytes).hexdigest()
after = HfApi(token=False).dataset_info(DATASET_ID, token=False)
after_names = {x.rfilename for x in after.siblings}
assert before_names == after_names, "A README-only update must not change dataset inventory"
print("PASS: Public dataset card refreshed; all original data files unchanged.")
print("https://huggingface.co/datasets/" + DATASET_ID)

# Safely remove ONLY the unrelated, never-populated placeholder model repo.
# This is NOT the model collection and is not one of the trained checkpoints.
EMPTY_PLACEHOLDER_ID = "i-am-shaurya05/shadowwork-shift-models"
MODEL_COLLECTION = (
    "i-am-shaurya05/shadowwork-shift-models-and-frozen-adapters-"
    "6af5c8a5a548fb703ab420ac"
)
registered_models = {
    item.item_id
    for item in api.get_collection(MODEL_COLLECTION).items
    if item.item_type == "model"
}
assert EMPTY_PLACEHOLDER_ID not in registered_models, (
    "Refusing to delete a model referenced by the public model collection."
)
empty_info = api.model_info(EMPTY_PLACEHOLDER_ID)
empty_files = {item.rfilename for item in empty_info.siblings}
assert empty_files == {".gitattributes"}, (
    "Refusing to delete a repository containing files other than .gitattributes.",
    sorted(empty_files),
)
api.delete_repo(repo_id=EMPTY_PLACEHOLDER_ID, repo_type="model")
print("Deleted confirmed empty placeholder:", EMPTY_PLACEHOLDER_ID)
try:
    api.model_info(EMPTY_PLACEHOLDER_ID, token=False)
except Exception as exc:
    print("Confirmed placeholder no longer accessible:", type(exc).__name__)
else:
    raise AssertionError("Deletion not verified: placeholder repository still exists.")
print("PASS: Dataset card synchronized; empty placeholder deleted; no model weights modified.")

